# Import library

In [10]:
import duckdb
import pandas as pd
from pathlib import Path

## Project folders

In [9]:
RAW_DIR = Path("../data/raw")
PARQUET_DIR = Path("../data/parquet")
PROCESSED_DIR = Path("../data/processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Columns and fiters

#### Check which monthly files exist

In [12]:
for file in sorted(RAW_DIR.glob("rideshare_2025_*.csv")):
    size_gb = file.stat().st_size / (1024**3)
    print(f"{file.name}: {size_gb:.2f} GB")

rideshare_2025_01.csv: 0.95 GB
rideshare_2025_02.csv: 0.94 GB


### Inspect January without loading the whole file

In [13]:
january_file = RAW_DIR / "rideshare_2025_01.csv"

duckdb.sql(f"""
    SELECT *
    FROM read_csv_auto(
        '{january_file}',
        normalize_names = true
    )
    LIMIT 10
""").df()

,trip_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,pickup_community_area,dropoff_community_area,fare,tip,additional_charges,trip_total,pickup_centroid_location,dropoff_centroid_location
0,fe7ff5fe55ee3afeb6f2414d656e7e7069740d19,2025-01-31 23:45:00,2025-02-01 00:00:00,831,3.1,17031242200,24,8,$7.5,$0,$4.5,$12,POINT (-87.6747191345 41.8995897957),POINT (-87.6207628651 41.8983317935)
1,fe9dff8b62ee79c06dceac0c65e4821cde73d72b,2025-01-31 23:45:00,2025-01-31 23:45:00,296,0.7,17031081800,8,8,$10,$0,$3.62,$13.62,POINT (-87.6378442095 41.8932163595),POINT (-87.6288741572 41.8920726347)
2,feba8a4b7128d9df374488dd63dd6e19904e4354,2025-01-31 23:45:00,2025-02-01 00:00:00,831,3.0,17031833000,28,22,$12.5,$0,$4.43,$16.93,POINT (-87.6572331997 41.8852813201),POINT (-87.689951469 41.9216471318)
3,ff05c8b66b2090704c9f197c5d384fde829725c1,2025-01-31 23:45:00,2025-02-01 00:00:00,749,2.2,17031081201,8,32,$15,$1,$4.34,$20.34,POINT (-87.6262105324 41.8991556134),POINT (-87.6314065252 41.8710158803)
4,ff0d0e86b289312c10b8e6e0c8f4c7ebe92c6d35,2025-01-31 23:45:00,2025-02-01 00:00:00,669,3.2,17031010202,1,<NA>,$7.5,$1,$3.09,$11.59,POINT (-87.6733337784 42.0160464924),None
5,ff3f9970d1fcc17bcbc6de68be6a39da2bb46190,2025-01-31 23:45:00,2025-01-31 23:45:00,326,0.8,17031838300,8,8,$12.5,$0,$1.23,$13.73,POINT (-87.6384040116 41.9015669095),POINT (-87.6266589003 41.9075200747)
6,ff38272a4b797a108425a1d462948e15bb812f57,2025-01-31 23:45:00,2025-02-01 00:00:00,566,2.3,17031833000,28,24,$10,$5,$4.2,$19.2,POINT (-87.6572331997 41.8852813201),POINT (-87.6709450752 41.9083786693)
7,ff53248ee4b0dac203007d7cae238d5e5eef986b,2025-01-31 23:45:00,2025-01-31 23:45:00,387,0.9,17031070101,7,7,$5,$0,$1.95,$6.95,POINT (-87.6358909539 41.9292629902),POINT (-87.6462109769 41.9218549112)
8,ff90e10368b87c9ae196385dd6c8e5ae0a1bb5ee,2025-01-31 23:45:00,2025-01-31 23:45:00,289,1.2,17031081700,8,28,$15,$0,$3.44,$18.44,POINT (-87.6318639497 41.8920421365),POINT (-87.6572331997 41.8852813201)
9,ff94c3773f9d50d0ea240d1903997c8c2493183c,2025-01-31 23:45:00,2025-01-31 23:45:00,380,1.4,17031062700,6,6,$7.5,$0,$1.93,$9.43,POINT (-87.6661106945 41.9360865352),POINT (-87.6430228044 41.9345397158)


### Check columns name

In [14]:
duckdb.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_csv_auto(
        '{january_file}',
        normalize_names = true
    )
""").df()

,column_name,column_type,null,key,default,extra
0,trip_id,VARCHAR,YES,None,None,None
1,trip_start_timestamp,TIMESTAMP,YES,None,None,None
2,trip_end_timestamp,TIMESTAMP,YES,None,None,None
3,trip_seconds,VARCHAR,YES,None,None,None
4,trip_miles,DOUBLE,YES,None,None,None
5,pickup_census_tract,BIGINT,YES,None,None,None
6,pickup_community_area,BIGINT,YES,None,None,None
7,dropoff_community_area,BIGINT,YES,None,None,None
8,fare,VARCHAR,YES,None,None,None
9,tip,VARCHAR,YES,None,None,None


### Check january row count

In [16]:
january_count = duckdb.sql(f"""
    SELECT COUNT(*)
    FROM read_csv_auto(
        '{january_file}',
        normalize_names = true
    )
""").fetchone()[0]

print(f"January rows: {january_count:,}")

January rows: 4,404,411


## Function to convert one month to Parquet

In [24]:
def convert_month_to_parquet(year, month):

    # File paths
    csv_file = RAW_DIR / f"rideshare_{year}_{month:02d}.csv"
    parquet_file = PARQUET_DIR / f"rideshare_{year}_{month:02d}.parquet"

    # Check CSV exists
    if not csv_file.exists():
        print(f"File not found: {csv_file}")
        return

    print(f"\nConverting {year}-{month:02d}...")

    # Remove old Parquet file if it exists
    if parquet_file.exists():
        parquet_file.unlink()

    # Convert CSV to Parquet
    # Keep all original values as strings for now
    duckdb.sql(f"""
        COPY (
            SELECT *
            FROM read_csv_auto(
                '{csv_file}',
                normalize_names = true,
                all_varchar = true
            )
        )
        TO '{parquet_file}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    print(f"Saved: {parquet_file}")
    print("Finished.")

### Convert Every month to parquet

In [28]:
for file in sorted(RAW_DIR.glob("rideshare_2025_*.csv")):

    year = int(file.stem.split("_")[1])
    month = int(file.stem.split("_")[2])

    convert_month_to_parquet(year, month)


Converting 2025-01...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: ../data/parquet/rideshare_2025_01.parquet
Finished.

Converting 2025-02...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: ../data/parquet/rideshare_2025_02.parquet
Finished.

Converting 2025-03...


...

Saved: ../data/parquet/rideshare_2025_10.parquet
Finished.

Converting 2025-11...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: ../data/parquet/rideshare_2025_11.parquet
Finished.

Converting 2025-12...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: ../data/parquet/rideshare_2025_12.parquet
Finished.
